In [1]:
# CELL 1 — Upload raw data files
from google.colab import files
import os
os.makedirs('/content/data/raw', exist_ok=True)
os.makedirs('/content/data/generated', exist_ok=True)
print('Upload all 9 raw files: users.csv, sessions.csv, orders.csv, order_items.csv, payments.csv, shipments.csv, refunds.csv, coupons.csv, products.json')
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f'/content/data/raw/{fname}', 'wb') as f:
        f.write(content)
    print(f'  Saved: {fname}')
print('All files uploaded!')

Upload all 9 raw files: users.csv, sessions.csv, orders.csv, order_items.csv, payments.csv, shipments.csv, refunds.csv, coupons.csv, products.json


Saving coupons.csv to coupons.csv
Saving meta.json to meta.json
Saving order_items.csv to order_items.csv
Saving orders.csv to orders.csv
Saving payments.csv to payments.csv
Saving products.json to products.json
Saving refunds.csv to refunds.csv
Saving sessions.csv to sessions.csv
Saving shipments.csv to shipments.csv
Saving users.csv to users.csv
  Saved: coupons.csv
  Saved: meta.json
  Saved: order_items.csv
  Saved: orders.csv
  Saved: payments.csv
  Saved: products.json
  Saved: refunds.csv
  Saved: sessions.csv
  Saved: shipments.csv
  Saved: users.csv
All files uploaded!


In [ ]:
# CELL 2 — Complete ETL Pipeline
import warnings, json
import numpy as np
import pandas as pd
from scipy import stats
warnings.filterwarnings('ignore')

RAW = '/content/data/raw'
OUT = '/content/data/generated'

# ── HELPERS ──────────────────────────────────────
def clean(df, name, pk=None, dates=None):
    for c in df.select_dtypes('object').columns:
        df[c] = df[c].astype(str).str.strip().str.lower()
        df[c] = df[c].replace({'nan':np.nan,'none':np.nan,'':np.nan})
    for c in (dates or []):
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce')
    n0 = len(df)
    df = df.drop_duplicates()
    if pk and pk in df.columns:
        df = df.drop_duplicates(subset=[pk], keep='last')
    print(f'  {name:15s}: {n0} -> {len(df)} rows')
    return df.reset_index(drop=True)

def to_num(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

# ── STEP 1: LOAD ──────────────────────────────────
print('=== STEP 1: LOAD ===')
users       = pd.read_csv(f'{RAW}/users.csv')
sessions    = pd.read_csv(f'{RAW}/sessions.csv')
orders      = pd.read_csv(f'{RAW}/orders.csv')
order_items = pd.read_csv(f'{RAW}/order_items.csv')
payments    = pd.read_csv(f'{RAW}/payments.csv')
shipments   = pd.read_csv(f'{RAW}/shipments.csv')
refunds     = pd.read_csv(f'{RAW}/refunds.csv')
coupons     = pd.read_csv(f'{RAW}/coupons.csv')
with open(f'{RAW}/products.json') as f:
    raw_p = json.load(f)
products = pd.DataFrame(raw_p if isinstance(raw_p, list) else [raw_p])
for n,d in [('users',users),('sessions',sessions),('orders',orders),
            ('order_items',order_items),('payments',payments),
            ('shipments',shipments),('refunds',refunds),
            ('coupons',coupons),('products',products)]:
    print(f'  {n:15s}: {d.shape}')

# ── STEP 2: CLEAN ─────────────────────────────────
print('\n=== STEP 2: CLEAN ===')
users       = clean(users,       'users',       pk='user_id',     dates=['signup_date','created_at'])
sessions    = clean(sessions,    'sessions',    pk='session_id',  dates=['session_start','session_end','session_date'])
orders      = clean(orders,      'orders',      pk='order_id',    dates=['order_ts','order_date','created_at'])
order_items = clean(order_items, 'order_items',                   dates=[])
payments    = clean(payments,    'payments',    pk='payment_id',  dates=['payment_ts','created_at'])
shipments   = clean(shipments,   'shipments',   pk='shipment_id', dates=['shipped_at','delivered_at','rto_at'])
refunds     = clean(refunds,     'refunds',     pk='refund_id',   dates=['refund_date','created_at'])
coupons     = clean(coupons,     'coupons',     pk='coupon_id',   dates=['expiry_date'])
products    = clean(products,    'products',    pk='product_id')
orders      = to_num(orders,      ['gross_amount','discount_amount','net_amount'])
order_items = to_num(order_items, ['unit_price','quantity'])
refunds     = to_num(refunds,     ['refund_amount'])

# ── STEP 3: FEATURES ──────────────────────────────
print('\n=== STEP 3: FEATURES ===')

# 3a payment failures
pay = payments.copy()
pay['status'] = pay['status'].fillna('unknown')
if 'payment_ts' in pay.columns:
    pay = pay.sort_values(['order_id','payment_ts'])
def fail_count(g):
    s = list(g['status'])
    return s.index('success') if 'success' in s else len(s)
pay_feat = pay.groupby('order_id').apply(fail_count).reset_index()
pay_feat.columns = ['order_id','payment_fail_count_before_success']
print(f'  pay_feat: {pay_feat.shape}')

# 3b device reuse
if 'device_id' in sessions.columns and 'user_id' in sessions.columns:
    dev_reuse = (sessions.groupby('device_id')['user_id']
                 .nunique().reset_index()
                 .rename(columns={'user_id':'device_reuse_count'}))
    scols = ['session_id','device_id']
    for c in ['channel','device_type']:
        if c in sessions.columns: scols.append(c)
    sess_feat = sessions[scols].drop_duplicates('session_id')
    sess_feat = sess_feat.merge(dev_reuse, on='device_id', how='left').drop(columns=['device_id'])
else:
    sess_feat = pd.DataFrame()
print(f'  sess_feat: {sess_feat.shape}')

# 3c pincode reuse
if 'shipping_pincode' in orders.columns and 'user_id' in orders.columns:
    pin_reuse = (orders.groupby('shipping_pincode')['user_id']
                 .nunique().reset_index()
                 .rename(columns={'user_id':'pincode_reuse_count'}))
else:
    pin_reuse = pd.DataFrame(columns=['shipping_pincode','pincode_reuse_count'])
print(f'  pin_reuse: {pin_reuse.shape}')

# 3d item features
item_agg = {}
if 'product_id' in order_items.columns:
    item_agg['item_count'] = ('product_id','count')
if 'quantity' in order_items.columns:
    item_agg['total_qty'] = ('quantity','sum')
if item_agg:
    item_feat = order_items.groupby('order_id').agg(**item_agg).reset_index()
else:
    item_feat = order_items[['order_id']].drop_duplicates()
if 'category' in order_items.columns and 'quantity' in order_items.columns:
    top_cat = (order_items.groupby(['order_id','category'])['quantity']
               .sum().reset_index().sort_values('quantity',ascending=False)
               .drop_duplicates('order_id')[['order_id','category']]
               .rename(columns={'category':'top_category'}))
    item_feat = item_feat.merge(top_cat, on='order_id', how='left')
print(f'  item_feat: {item_feat.shape}')

# 3e refund features
ref = refunds.copy()
if 'status' in ref.columns:
    ref = ref[ref['status'].str.contains('approved|success|done', na=False)]
ref_feat = ref.groupby('order_id').agg(refund_amount=('refund_amount','sum')).reset_index()
ref_feat['refund_approved'] = 1
print(f'  ref_feat: {ref_feat.shape}')

# 3f RTO
ship = shipments.copy()
if 'status' in ship.columns:
    ship['rto_flag'] = ship['status'].str.contains('rto', na=False).astype(int)
elif 'rto_at' in ship.columns:
    ship['rto_flag'] = ship['rto_at'].notna().astype(int)
else:
    ship['rto_flag'] = 0
rto_feat = ship.groupby('order_id')['rto_flag'].max().reset_index()
print(f'  rto_feat: {rto_feat.shape}')

# 3g coupon features
o = orders[['order_id','user_id','gross_amount','discount_amount']].copy()
if 'coupon_id' in orders.columns:
    o['coupon_id'] = orders['coupon_id']
o['coupon_discount_pct'] = np.where(
    o['gross_amount'] > 0,
    (o['discount_amount'].fillna(0) / o['gross_amount'] * 100).round(2), 0)
if 'coupon_id' in o.columns:
    coup_orders = o[o['coupon_id'].notna()]
    user_coup = (coup_orders.groupby('user_id')['order_id'].count()
                 .reset_index().rename(columns={'order_id':'user_coupon_count'}))
    o = o.merge(user_coup, on='user_id', how='left')
    o['user_coupon_count'] = o['user_coupon_count'].fillna(0)
    o['multi_coupon_user_flag'] = (o['user_coupon_count'] > 1).astype(int)
else:
    o['multi_coupon_user_flag'] = 0
coup_feat = o[['order_id','coupon_discount_pct','multi_coupon_user_flag']]
print(f'  coup_feat: {coup_feat.shape}')

=== STEP 1: LOAD ===
  users          : (3200, 7)
  sessions       : (104683, 9)
  orders         : (4637, 18)
  order_items    : (11533, 4)
  payments       : (6200, 8)
  shipments      : (4558, 7)
  refunds        : (531, 6)
  coupons        : (15, 9)
  products       : (260, 6)

=== STEP 2: CLEAN ===
  users          : 3200 -> 3200 rows
  sessions       : 104683 -> 104422 rows
  orders         : 4637 -> 4626 rows
  order_items    : 11533 -> 11510 rows
  payments       : 6200 -> 6182 rows
  shipments      : 4558 -> 4549 rows
  refunds        : 531 -> 530 rows
  coupons        : 15 -> 14 rows
  products       : 260 -> 260 rows

=== STEP 3: FEATURES ===


KeyError: 'status'

In [ ]:
# DEBUG — Check actual column names
print('payments columns:', list(payments.columns))
print('shipments columns:', list(shipments.columns))
print('refunds columns:', list(refunds.columns))
print('orders columns:', list(orders.columns))

payments columns: ['payment_id', 'order_id', 'attempt_no', 'payment_ts', 'payment_method', 'payment_status', 'failure_reason', 'amount']
shipments columns: ['shipment_id', 'order_id', 'carrier', 'shipped_ts', 'delivered_ts', 'rto_ts', 'shipment_status']
refunds columns: ['refund_id', 'order_id', 'refund_ts', 'refund_amount', 'refund_reason', 'refund_status']
orders columns: ['order_id', 'session_id', 'user_id', 'order_ts', 'channel', 'device', 'device_id', 'ip_hash', 'shipping_pincode', 'city_tier', 'payment_method', 'coupon_id', 'gross_amount', 'discount_amount', 'shipping_amount', 'net_amount', 'promo_flag', 'order_status']


In [ ]:
# FIXED CELL — Steps 3, 4, 5 with correct column names
import warnings, json
import numpy as np
import pandas as pd
from scipy import stats
warnings.filterwarnings('ignore')

OUT = '/content/data/generated'

print('=== STEP 3: FEATURES (FIXED) ===')

# 3a payment failures (correct col: payment_status)
pay = payments.copy()
pay['payment_status'] = pay['payment_status'].fillna('unknown')
if 'payment_ts' in pay.columns:
    pay = pay.sort_values(['order_id','payment_ts'])

def fail_count(g):
    s = list(g['payment_status'])
    return s.index('success') if 'success' in s else len(s)

pay_feat = pay.groupby('order_id').apply(fail_count).reset_index()
pay_feat.columns = ['order_id','payment_fail_count_before_success']
print(f'  pay_feat: {pay_feat.shape}')

# 3b device reuse
if 'device_id' in sessions.columns and 'user_id' in sessions.columns:
    dev_reuse = (sessions.groupby('device_id')['user_id']
                 .nunique().reset_index()
                 .rename(columns={'user_id':'device_reuse_count'}))
    scols = ['session_id','device_id']
    for c in ['channel','device_type','device']:
        if c in sessions.columns: scols.append(c)
    sess_feat = sessions[scols].drop_duplicates('session_id')
    sess_feat = sess_feat.merge(dev_reuse, on='device_id', how='left').drop(columns=['device_id'])
else:
    sess_feat = pd.DataFrame()
print(f'  sess_feat: {sess_feat.shape}')

# 3c pincode reuse
pin_col = 'shipping_pincode' if 'shipping_pincode' in orders.columns else None
if not pin_col:
    for c in orders.columns:
        if 'pin' in c or 'zip' in c: pin_col = c; break
if pin_col and 'user_id' in orders.columns:
    pin_reuse = (orders.groupby(pin_col)['user_id']
                 .nunique().reset_index()
                 .rename(columns={'user_id':'pincode_reuse_count', pin_col:'shipping_pincode'}))
else:
    pin_reuse = pd.DataFrame(columns=['shipping_pincode','pincode_reuse_count'])
print(f'  pin_reuse: {pin_reuse.shape}')

# 3d item features
item_agg = {}
if 'product_id' in order_items.columns:
    item_agg['item_count'] = ('product_id','count')
if 'quantity' in order_items.columns:
    item_agg['total_qty'] = ('quantity','sum')
if item_agg:
    item_feat = order_items.groupby('order_id').agg(**item_agg).reset_index()
else:
    item_feat = order_items[['order_id']].drop_duplicates()
if 'category' in order_items.columns and 'quantity' in order_items.columns:
    top_cat = (order_items.groupby(['order_id','category'])['quantity']
               .sum().reset_index().sort_values('quantity',ascending=False)
               .drop_duplicates('order_id')[['order_id','category']]
               .rename(columns={'category':'top_category'}))
    item_feat = item_feat.merge(top_cat, on='order_id', how='left')
print(f'  item_feat: {item_feat.shape}')

# 3e refund features (correct col: refund_status)
ref = refunds.copy()
if 'refund_status' in ref.columns:
    ref = ref[ref['refund_status'].str.contains('approved|success|done|completed', na=False)]
ref_feat = ref.groupby('order_id').agg(refund_amount=('refund_amount','sum')).reset_index()
ref_feat['refund_approved'] = 1
print(f'  ref_feat: {ref_feat.shape}')

# 3f RTO (correct col: shipment_status, rto_ts)
ship = shipments.copy()
if 'shipment_status' in ship.columns:
    ship['rto_flag'] = ship['shipment_status'].str.contains('rto', na=False).astype(int)
elif 'rto_ts' in ship.columns:
    ship['rto_flag'] = ship['rto_ts'].notna().astype(int)
else:
    ship['rto_flag'] = 0
rto_feat = ship.groupby('order_id')['rto_flag'].max().reset_index()
print(f'  rto_feat: {rto_feat.shape}')

# 3g coupon features
amt_col = 'gross_amount' if 'gross_amount' in orders.columns else 'order_amount'
disc_col = 'discount_amount' if 'discount_amount' in orders.columns else None
o = orders[['order_id','user_id']].copy()
if amt_col in orders.columns: o[amt_col] = orders[amt_col]
if disc_col: o['discount_amount'] = orders[disc_col]
if 'coupon_id' in orders.columns: o['coupon_id'] = orders['coupon_id']

if disc_col and amt_col:
    o['coupon_discount_pct'] = np.where(
        o[amt_col] > 0,
        (o['discount_amount'].fillna(0) / o[amt_col] * 100).round(2), 0)
else:
    o['coupon_discount_pct'] = 0

if 'coupon_id' in o.columns:
    coup_orders = o[o['coupon_id'].notna()]
    user_coup = (coup_orders.groupby('user_id')['order_id'].count()
                 .reset_index().rename(columns={'order_id':'user_coupon_count'}))
    o = o.merge(user_coup, on='user_id', how='left')
    o['user_coupon_count'] = o['user_coupon_count'].fillna(0)
    o['multi_coupon_user_flag'] = (o['user_coupon_count'] > 1).astype(int)
else:
    o['multi_coupon_user_flag'] = 0
coup_feat = o[['order_id','coupon_discount_pct','multi_coupon_user_flag']]
print(f'  coup_feat: {coup_feat.shape}')

print('\n=== STEP 4: BUILD fact_orders_enriched ===')
fact = orders.copy()

if 'session_id' in fact.columns and not sess_feat.empty:
    fact = fact.merge(sess_feat, on='session_id', how='left')
fact = fact.merge(item_feat,  on='order_id', how='left')
fact = fact.merge(pay_feat,   on='order_id', how='left')
fact['payment_fail_count_before_success'] = fact['payment_fail_count_before_success'].fillna(0)
fact = fact.merge(ref_feat,   on='order_id', how='left')
fact['refund_amount']   = fact['refund_amount'].fillna(0)
fact['refund_approved'] = fact['refund_approved'].fillna(0).astype(int)
fact = fact.merge(rto_feat,   on='order_id', how='left')
fact['rto_flag'] = fact['rto_flag'].fillna(0).astype(int)
fact = fact.merge(coup_feat,  on='order_id', how='left')
fact['coupon_discount_pct']    = fact['coupon_discount_pct'].fillna(0)
fact['multi_coupon_user_flag'] = fact['multi_coupon_user_flag'].fillna(0).astype(int)

if 'shipping_pincode' in fact.columns and not pin_reuse.empty:
    fact = fact.merge(pin_reuse, on='shipping_pincode', how='left')
    fact['pincode_reuse_count'] = fact['pincode_reuse_count'].fillna(1)
else:
    fact['pincode_reuse_count'] = 1

fact['new_user_flag'] = 0
if 'signup_date' in users.columns:
    users2 = users[['user_id','signup_date']].copy()
    users2['signup_date'] = pd.to_datetime(users2['signup_date'], errors='coerce')
    fact = fact.merge(users2, on='user_id', how='left')
    fact['order_ts'] = pd.to_datetime(fact['order_ts'], errors='coerce')
    fact['days_since_signup'] = (fact['order_ts'] - fact['signup_date']).dt.days.fillna(999)
    fact['new_user_flag'] = (fact['days_since_signup'] <= 7).astype(int)

fact['cod_flag'] = 0
pay_method_col = 'payment_method' if 'payment_method' in fact.columns else None
if pay_method_col:
    fact['cod_flag'] = fact[pay_method_col].str.contains('cod|cash', na=False).astype(int)

if 'top_category' in fact.columns and 'gross_amount' in fact.columns:
    fact['order_value_zscore_by_category'] = (
        fact.groupby('top_category')['gross_amount']
        .transform(lambda x: stats.zscore(x, nan_policy='omit'))
    ).fillna(0).round(3)
else:
    fact['order_value_zscore_by_category'] = 0

if 'total_qty' in fact.columns:
    fact['qty_outlier_flag'] = (fact['total_qty'] > fact['total_qty'].quantile(0.97)).astype(int)
else:
    fact['qty_outlier_flag'] = 0

if 'shipping_city_tier' not in fact.columns:
    fact['shipping_city_tier'] = 'unknown'

print(f'  fact shape: {fact.shape}')

print('\n=== STEP 5: RISK SCORING ===')
fact['high_discount_flag'] = (fact['coupon_discount_pct'] > 40).astype(int)
score = pd.Series(0.0, index=fact.index)
score += fact['high_discount_flag'] * 12
score += fact['multi_coupon_user_flag'] * 10
score += (fact['payment_fail_count_before_success'] >= 2).astype(int) * 15
if 'device_reuse_count' in fact.columns:
    score += (fact['device_reuse_count'].fillna(0) >= 3).astype(int) * 12
score += (fact['pincode_reuse_count'] >= 4).astype(int) * 10
score += (fact['order_value_zscore_by_category'].abs() > 2.5).astype(int) * 10
score += fact['qty_outlier_flag'] * 8
score += fact['new_user_flag'] * 8
score += fact['cod_flag'] * 8
score += fact['rto_flag'] * 7
fact['risk_score'] = score.clip(0, 100).round(1)
fact['risk_band'] = pd.cut(fact['risk_score'], bins=[-1,34.9,59.9,100], labels=['Low','Medium','High'])
print(f'  Risk distribution:\n{fact["risk_band"].value_counts().to_string()}')
print(f'  fact_orders_enriched shape: {fact.shape}')

=== STEP 3: FEATURES (FIXED) ===
  pay_feat: (4626, 2)
  sess_feat: (104422, 4)
  pin_reuse: (690, 2)
  item_feat: (4626, 3)
  ref_feat: (530, 3)
  rto_feat: (4538, 2)
  coup_feat: (4626, 3)

=== STEP 4: BUILD fact_orders_enriched ===
  fact shape: (4626, 37)

=== STEP 5: RISK SCORING ===
  Risk distribution:
risk_band
Low       4528
Medium      98
High         0
  fact_orders_enriched shape: (4626, 40)


In [ ]:
# CELL 5 — Generate 3 Output CSVs + Download
from google.colab import files
import os
OUT = '/content/data/generated'
os.makedirs(OUT, exist_ok=True)

# OUTPUT 1: fact_orders_enriched.csv
print('=== OUTPUT 1: fact_orders_enriched.csv ===')
want = ['order_id','user_id','session_id','order_ts','gross_amount','discount_amount',
        'net_amount','payment_method','coupon_id','coupon_discount_pct','item_count',
        'total_qty','top_category','shipping_pincode','shipping_city_tier',
        'payment_fail_count_before_success','device_reuse_count','pincode_reuse_count',
        'order_value_zscore_by_category','qty_outlier_flag','multi_coupon_user_flag',
        'high_discount_flag','new_user_flag','cod_flag','refund_amount','refund_approved',
        'rto_flag','risk_score','risk_band']
cols1 = [c for c in want if c in fact.columns]
fact[cols1].to_csv(f'{OUT}/fact_orders_enriched.csv', index=False)
print(f'  Saved: {fact[cols1].shape}')

# OUTPUT 2: fact_user_risk_weekly.csv
print('\n=== OUTPUT 2: fact_user_risk_weekly.csv ===')
f2 = fact.copy()
f2['order_ts'] = pd.to_datetime(f2['order_ts'], errors='coerce')
f2['week_start'] = f2['order_ts'].dt.to_period('W').apply(lambda x: x.start_time)
weekly = f2.groupby(['user_id','week_start']).agg(
    orders_count           = ('order_id','count'),
    net_revenue            = ('net_amount','sum') if 'net_amount' in f2.columns else ('order_id','count'),
    refunds_count          = ('refund_approved','sum'),
    refund_amount          = ('refund_amount','sum'),
    coupon_orders_count    = ('multi_coupon_user_flag','sum'),
    avg_discount_pct       = ('coupon_discount_pct','mean'),
    cod_orders_count       = ('cod_flag','sum'),
    rto_count              = ('rto_flag','sum'),
    payment_failures_count = ('payment_fail_count_before_success','sum'),
    risk_score_avg         = ('risk_score','mean')
).reset_index()
weekly['risk_score_avg'] = weekly['risk_score_avg'].round(2)
weekly.to_csv(f'{OUT}/fact_user_risk_weekly.csv', index=False)
print(f'  Saved: {weekly.shape}')

# OUTPUT 3: investigation_queue.csv
print('\n=== OUTPUT 3: investigation_queue.csv ===')

def top3_reasons(row):
    r = []
    if row.get('payment_fail_count_before_success',0) >= 2:
        r.append(f'payment_failures={int(row["payment_fail_count_before_success"])}')
    if row.get('coupon_discount_pct',0) > 40:
        r.append(f'high_discount={row["coupon_discount_pct"]}%')
    if row.get('multi_coupon_user_flag',0) == 1:
        r.append('multi_coupon_abuse')
    if row.get('device_reuse_count',0) >= 3:
        r.append(f'device_reuse={int(row["device_reuse_count"])}')
    if row.get('pincode_reuse_count',0) >= 4:
        r.append(f'pincode_reuse={int(row["pincode_reuse_count"])}')
    if abs(row.get('order_value_zscore_by_category',0)) > 2.5:
        r.append(f'value_zscore={row["order_value_zscore_by_category"]}')
    if row.get('qty_outlier_flag',0) == 1:
        r.append('qty_outlier')
    if row.get('new_user_flag',0) == 1 and row.get('cod_flag',0) == 1:
        r.append('new_user_cod')
    if row.get('rto_flag',0) == 1:
        r.append('rto_history')
    return ' | '.join(r[:3]) if r else 'no_flag'

def action(band):
    return {'High':'hold_for_manual_review','Medium':'call_verification'}.get(str(band),'auto_approve')

queue = fact[fact['risk_score'] >= 35].copy().sort_values('risk_score', ascending=False).reset_index(drop=True)
queue['top_3_risk_reasons'] = queue.apply(top3_reasons, axis=1)
queue['recommended_action'] = queue['risk_band'].apply(action)
evid = ['coupon_discount_pct','payment_fail_count_before_success','device_reuse_count',
        'pincode_reuse_count','order_value_zscore_by_category','multi_coupon_user_flag',
        'new_user_flag','cod_flag','rto_flag']
qcols = ['order_id','user_id','risk_score','risk_band','top_3_risk_reasons','recommended_action'] + \
        [c for c in evid if c in queue.columns]
queue[qcols].to_csv(f'{OUT}/investigation_queue.csv', index=False)
print(f'  Saved: {queue[qcols].shape}')
print(f'  High: {(queue["risk_band"]=="High").sum()}  Medium: {(queue["risk_band"]=="Medium").sum()}')

print('\n==============================')
print('ALL 3 OUTPUT FILES READY!')
print('==============================')
files.download(f'{OUT}/fact_orders_enriched.csv')
files.download(f'{OUT}/fact_user_risk_weekly.csv')
files.download(f'{OUT}/investigation_queue.csv')

=== OUTPUT 1: fact_orders_enriched.csv ===
  Saved: (4626, 28)

=== OUTPUT 2: fact_user_risk_weekly.csv ===
  Saved: (3816, 12)

=== OUTPUT 3: investigation_queue.csv ===
  Saved: (98, 15)
  High: 0  Medium: 98

ALL 3 OUTPUT FILES READY!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>